## 1. Inference

In [ ]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Substitua pelos imports corretos da biblioteca do SAM que você está utilizando
from sam3 import sam_model_registry, SamPredictor

# 1. Configurações Iniciais
# Verifique no seu config_resolved.yaml qual foi a arquitetura base (vit_b, vit_l, vit_h)
model_type = "vit_b" 
checkpoint_path = "/home/avmoura_linux/Documents/unb/sandbox_sam3/logs/ph2_train_seg_500_3/checkpoints/checkpoint.pt" # ou checkpoint_500.pt
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. Carregar o Modelo
print(f"Carregando modelo no dispositivo: {device}")
sam = sam_model_registry[model_type](checkpoint=checkpoint_path)
sam.to(device=device)

predictor = SamPredictor(sam)

# 3. Carregar e Preparar a Imagem (Exemplo com uma imagem do PH2)
image_path = "/home/avmoura_linux/Documents/unb/sandbox_sam3/datasets/ph2_dataset/test/IMD146_bmp.rf.3e4494d39853b6bb0da42ae0b41105c4.jpg" # PH2 geralmente usa .bmp
image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

predictor.set_image(image)

# 4. Definir o Prompt e Inferir
# Como o SAM é "promptable", para lesões de pele o ideal costuma ser um Bounding Box
# ou um ponto central na lesão. Vamos usar um ponto central como exemplo.
altura, largura, _ = image.shape
input_point = np.array([[largura // 2, altura // 2]]) # Ponto no centro da imagem
input_label = np.array([1]) # 1 indica que queremos a máscara do objeto (foreground)

# Predição
masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=False, # Retorna apenas a melhor máscara
)

# 5. Visualizar o Resultado
def show_mask(mask, ax, random_color=False):
    if random_color:
        color = np.concatenate([np.random.random(3), np.array([0.6])], axis=0)
    else:
        color = np.array([30/255, 144/255, 255/255, 0.6])
    h, w = mask.shape[-2:]
    mask_image = mask.reshape(h, w, 1) * color.reshape(1, 1, -1)
    ax.imshow(mask_image)

plt.figure(figsize=(10, 10))
plt.imshow(image)
show_mask(masks[0], plt.gca())
plt.title(f"Score da Máscara: {scores[0]:.3f}", fontsize=14)
plt.axis('off')
plt.show()

AttributeError: partially initialized module 'torch' has no attribute 'library' (most likely due to a circular import)